# mfd_sellamt_d因子

mfd_sellamt_d 的含义是：某只股票在某个交易日内，主力资金流出的成交金额。

## 市值行业中性化后因子指标计算

由于该因子很显然会与市值存在相关性，然而该因子所要表达的信息却又和市值行业不相关，于是直接做市值行业中性化后再对因子的指标进行计算

In [ ]:
# -*- coding: utf-8 -*-
"""
BigQuant 复现华泰资金流向因子 mfd_sellamt_d：主力流出额（市值行业中性化版本）。

口径：
1. mfd_sellamt_d ≈ cn_stock_moneyflow.outflow_amount_main。
2. 流出类因子为反向因子，原始因子值取 -outflow_amount_main。
3. 每个截面对原始因子做 MAD 去极值、标准化，再对 log(流通市值) 和行业哑变量做中性化，最后对中性化残差再次标准化。
4. 以 10 个交易日作为截面周期，使用中性化后的因子计算未来 10 个交易日收益对应的 IC、RankIC、回归因子收益率和 t 值。
5. 股票池剔除 ST、当前停牌、下一交易日停牌；使用 cn_stock_factors_base 的后复权 close 计算收益。
6. 回归法参考研报：未来 10 日相对沪深300超额收益 ~ 行业哑变量 + 中性化后因子，WLS 权重为 sqrt(流通市值)。

优化要点：
1. 在 DAI SQL 端完成交易日抽样、未来收益、停牌过滤和基准收益计算，只把调仓截面传回 Python。
2. Python 端只保留必要列，并将 instrument/industry 转为 category，数值列转为 float32，降低内存占用。
3. 截面中性化和截面回归全部使用 NumPy 矩阵运算，避免逐截面反复创建 statsmodels 对象。
4. 不输出日志，只 display 汇总表，并画 IC 与 RankIC 同图。
"""

import os
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display

try:
    import dai
except ImportError:
    from bigquant import dai  # type: ignore

warnings.filterwarnings("ignore")


@dataclass(frozen=True)
class Config:
    start_date: str = "2020-01-01"       # cn_stock_moneyflow 官网数据起点为 2015-01-01
    end_date: str = "2026-06-30"
    forward_days: int = 10
    rebalance_freq: int = 10
    min_cross_section_size: int = 30
    industry_col: str = "sw2021_level1"  # 可改 sw2014_level1；取决于账号内 cn_stock_factors_base 字段
    factor_name: str = "mfd_sellamt_d_neutral"
    factor_source_col: str = "outflow_amount_main"
    chinese_font_path: str = ""          # 若 BigQuant 环境仍无法显示中文，可上传中文字体文件后填入绝对路径


CFG = Config()


def _font_has_chinese(font_path: str) -> bool:
    """粗略判断字体是否包含常用中文字形，避免误选英文字体。"""
    try:
        ft = font_manager.get_font(font_path)
        cmap = ft.get_charmap()
        return all(ord(ch) in cmap for ch in "因子日期相关系数")
    except Exception:
        return False


def _candidate_font_paths(user_font_path: str = "") -> List[str]:
    """优先使用用户指定字体，其次在 BigQuant/Jupyter/Linux/Windows/macOS 常见路径中搜索中文字体。"""
    candidates: List[str] = []

    if user_font_path:
        candidates.append(user_font_path)

    env_font = os.environ.get("CHINESE_FONT_PATH", "")
    if env_font:
        candidates.append(env_font)

    direct_paths = [
        "./SimHei.ttf",
        "./simhei.ttf",
        "./msyh.ttc",
        "./Microsoft YaHei.ttf",
        "./NotoSansCJK-Regular.ttc",
        "./NotoSansCJKsc-Regular.otf",
        "/home/jovyan/work/SimHei.ttf",
        "/home/jovyan/work/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.otf",
        "/usr/share/fonts/opentype/noto/NotoSansCJKsc-Regular.otf",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/arphic/uming.ttc",
        "/System/Library/Fonts/PingFang.ttc",
        "C:/Windows/Fonts/msyh.ttc",
        "C:/Windows/Fonts/simhei.ttf",
    ]
    candidates.extend(direct_paths)

    search_roots = [
        Path.cwd(),
        Path.home(),
        Path("/usr/share/fonts"),
        Path("/usr/local/share/fonts"),
        Path("/opt/conda/lib/python3.10/site-packages/matplotlib/mpl-data/fonts/ttf"),
    ]
    name_keywords = (
        "NotoSansCJK",
        "NotoSansSC",
        "SourceHanSans",
        "SourceHanSerif",
        "WenQuanYi",
        "wqy",
        "SimHei",
        "simhei",
        "msyh",
        "PingFang",
        "Arial Unicode",
    )
    suffixes = {".ttf", ".ttc", ".otf"}

    for root in search_roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob("*"):
                if p.suffix.lower() in suffixes and any(k.lower() in p.name.lower() for k in name_keywords):
                    candidates.append(str(p))
        except Exception:
            continue

    seen = set()
    unique = []
    for p in candidates:
        pp = str(Path(p).expanduser())
        if pp not in seen:
            seen.add(pp)
            unique.append(pp)
    return unique


def set_chinese_font(font_path: str = "") -> Optional[font_manager.FontProperties]:
    """设置 Matplotlib 中文字体，并返回可显式传入标题/坐标轴/图例的 FontProperties。"""
    preferred_names = [
        "Microsoft YaHei",
        "SimHei",
        "Noto Sans CJK SC",
        "Noto Sans SC",
        "Source Han Sans SC",
        "WenQuanYi Micro Hei",
        "PingFang SC",
        "Arial Unicode MS",
    ]

    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42
    plt.rcParams["svg.fonttype"] = "none"

    for path in _candidate_font_paths(font_path):
        if not Path(path).exists():
            continue
        if not _font_has_chinese(path):
            continue
        try:
            font_manager.fontManager.addfont(path)
            prop = font_manager.FontProperties(fname=path)
            font_name = prop.get_name()
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [font_name] + preferred_names + ["DejaVu Sans"]
            return prop
        except Exception:
            continue

    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in preferred_names:
        if name in available:
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [name] + [n for n in preferred_names if n != name] + ["DejaVu Sans"]
            return font_manager.FontProperties(family=name)

    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = preferred_names + ["DejaVu Sans"]
    warnings.warn(
        "当前运行环境未找到可用中文字体。若图表仍显示方框，请上传 SimHei.ttf、msyh.ttc "
        "或 NotoSansCJK-Regular.ttc，并在 CFG.chinese_font_path 中填入该字体路径。",
        RuntimeWarning,
    )
    return None


def fetch_signal_panel(cfg: Config) -> pd.DataFrame:
    """只读取调仓截面所需数据，减少传输量和内存占用。"""
    fetch_end = (pd.Timestamp(cfg.end_date) + pd.Timedelta(days=max(90, cfg.forward_days * 12))).strftime("%Y-%m-%d")

    sql = f"""
    PRAGMA enable_pushdown_window;

    WITH trading_dates AS (
        SELECT
            date,
            ROW_NUMBER() OVER (ORDER BY date) AS rn
        FROM (
            SELECT DISTINCT date
            FROM cn_stock_factors_base
            WHERE date >= DATE '{cfg.start_date}'
              AND date <= DATE '{cfg.end_date}'
        )
    ),

    signal_dates AS (
        SELECT date
        FROM trading_dates
        WHERE MOD(rn - 1, {cfg.rebalance_freq}) = 0
    ),

    base AS (
        SELECT
            date,
            instrument,
            close,
            float_market_cap,
            {cfg.industry_col} AS industry_level1,
            st_status,
            suspended,
            list_sector,
            LEAD(close, {cfg.forward_days}) OVER (PARTITION BY instrument ORDER BY date) AS close_fwd,
            LEAD(suspended, 1) OVER (PARTITION BY instrument ORDER BY date) AS next_suspended
        FROM cn_stock_factors_base
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{fetch_end}'
          AND list_sector != 4
    ),

    bench_raw AS (
        SELECT
            date,
            MAX(hs300_close) AS hs300_close
        FROM cn_stock_factors_base
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{fetch_end}'
        GROUP BY date
    ),

    bench AS (
        SELECT
            date,
            hs300_close,
            LEAD(hs300_close, {cfg.forward_days}) OVER (ORDER BY date) AS hs300_close_fwd
        FROM bench_raw
    )

    SELECT
        b.date,
        b.instrument,
        b.float_market_cap,
        b.industry_level1,
        -mf.{cfg.factor_source_col} AS factor_raw,
        b.close_fwd / b.close - 1.0 AS ret_fwd_10d,
        b.close_fwd / b.close - 1.0 - (be.hs300_close_fwd / be.hs300_close - 1.0) AS excess_ret_fwd_10d
    FROM base AS b
    JOIN signal_dates AS sd
      ON b.date = sd.date
    JOIN cn_stock_moneyflow AS mf
      ON b.date = mf.date AND b.instrument = mf.instrument
    JOIN bench AS be
      ON b.date = be.date
    WHERE b.date <= DATE '{cfg.end_date}'
      AND b.st_status = 0
      AND b.suspended = 0
      AND COALESCE(b.next_suspended, 1) = 0
      AND b.close > 0
      AND b.close_fwd > 0
      AND be.hs300_close > 0
      AND be.hs300_close_fwd > 0
      AND b.float_market_cap > 0
      AND mf.{cfg.factor_source_col} IS NOT NULL
    ORDER BY b.date, b.instrument
    """

    df = dai.query(sql, filters={"date": [cfg.start_date, fetch_end]}).df()
    if df.empty:
        raise ValueError("查询结果为空，请检查日期区间、权限或字段名。")

    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype("category")
    df["industry_level1"] = df["industry_level1"].fillna("未知").astype("category")

    for col in ["float_market_cap", "factor_raw", "ret_fwd_10d", "excess_ret_fwd_10d"]:
        df[col] = pd.to_numeric(df[col], errors="coerce", downcast="float")

    df = df.dropna(subset=["float_market_cap", "factor_raw", "ret_fwd_10d", "excess_ret_fwd_10d"])
    return df.reset_index(drop=True)


def robust_zscore_np(x: np.ndarray) -> np.ndarray:
    """MAD 去极值后标准化。"""
    x = x.astype(np.float64, copy=False)
    out = np.full(x.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(x)
    if valid.sum() < 3:
        return out

    xv = x[valid]
    med = np.nanmedian(xv)
    mad = np.nanmedian(np.abs(xv - med))
    if np.isfinite(mad) and mad > 1e-12:
        scale = 1.4826 * mad
        lo, hi = med - 3.0 * scale, med + 3.0 * scale
    else:
        lo, hi = np.nanpercentile(xv, [1.0, 99.0])

    xv = np.clip(xv, lo, hi)
    std = xv.std(ddof=0)
    if np.isfinite(std) and std > 1e-12:
        out[valid] = (xv - xv.mean()) / std
    return out


def _neutralize_array_by_cap_industry(
    y: np.ndarray,
    float_market_cap: np.ndarray,
    industry: pd.Series,
) -> np.ndarray:
    """y 对 log(流通市值) 和行业哑变量做截面中性化，返回残差。"""
    y = y.astype(np.float64, copy=False)
    log_cap = np.log(float_market_cap.astype(np.float64, copy=False).clip(min=1.0))
    dummies = industry_dummies(industry)
    X = np.column_stack([np.ones(len(y), dtype=np.float64), log_cap, dummies])

    valid = np.isfinite(y) & np.isfinite(X).all(axis=1)
    resid = np.full(len(y), np.nan, dtype=np.float64)
    if valid.sum() < max(10, X.shape[1] + 2):
        return resid

    Xv = X[valid]
    yv = y[valid]
    beta = np.linalg.lstsq(Xv, yv, rcond=None)[0]
    resid[valid] = yv - Xv @ beta
    return resid


def add_neutralized_factor(df: pd.DataFrame) -> pd.DataFrame:
    """
    构造最终测试因子：
    1. 原始流出类因子取反：factor_raw = -outflow_amount_main；
    2. 每个调仓截面对 factor_raw 做 MAD 去极值和标准化；
    3. 将标准化后的因子对 log(流通市值) 和行业哑变量做中性化；
    4. 对中性化残差再次做 MAD 去极值和标准化，得到 factor_neutral_z。
    """
    df = df.copy()
    n = len(df)
    factor_z = np.full(n, np.nan, dtype=np.float32)
    factor_neutral = np.full(n, np.nan, dtype=np.float32)
    factor_neutral_z = np.full(n, np.nan, dtype=np.float32)

    raw_values = df["factor_raw"].to_numpy(dtype=np.float64, copy=False)
    cap_values = df["float_market_cap"].to_numpy(dtype=np.float64, copy=False)

    for _, idx in df.groupby("date", sort=False, observed=True).indices.items():
        idx_arr = np.asarray(idx)

        z = robust_zscore_np(raw_values[idx_arr])
        factor_z[idx_arr] = z.astype(np.float32)

        resid = _neutralize_array_by_cap_industry(
            y=z,
            float_market_cap=cap_values[idx_arr],
            industry=df["industry_level1"].iloc[idx_arr],
        )
        factor_neutral[idx_arr] = resid.astype(np.float32)

        # 中性化残差再标准化，保证最终因子在每个截面上可比较。
        neutral_z = robust_zscore_np(resid)
        factor_neutral_z[idx_arr] = neutral_z.astype(np.float32)

    df["factor_z_before_neutral"] = factor_z
    df["factor_neutral"] = factor_neutral
    df["factor_neutral_z"] = factor_neutral_z
    df = df.dropna(subset=["factor_neutral_z"]).reset_index(drop=True)
    return df


def corr_np(x: np.ndarray, y: np.ndarray) -> float:
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3:
        return np.nan
    xv = x[valid].astype(np.float64, copy=False)
    yv = y[valid].astype(np.float64, copy=False)
    xv = xv - xv.mean()
    yv = yv - yv.mean()
    denom = np.sqrt(np.dot(xv, xv) * np.dot(yv, yv))
    if not np.isfinite(denom) or denom <= 1e-18:
        return np.nan
    return float(np.dot(xv, yv) / denom)


def rank_np(x: np.ndarray) -> np.ndarray:
    return pd.Series(x).rank(method="average").to_numpy(dtype=np.float64, copy=False)


def industry_dummies(industry: pd.Series) -> np.ndarray:
    """行业哑变量，drop_first=True；返回 float64 矩阵。"""
    codes = pd.Categorical(industry).codes
    n = len(codes)
    k = int(codes.max()) + 1
    if k <= 1:
        return np.empty((n, 0), dtype=np.float64)

    mat = np.zeros((n, k - 1), dtype=np.float64)
    rows = np.arange(n)
    mask = codes > 0
    mat[rows[mask], codes[mask] - 1] = 1.0
    return mat


def wls_factor_return(g: pd.DataFrame) -> Tuple[float, float]:
    """未来 10 日超额收益 ~ 中性化后因子 + 行业哑变量；WLS 权重为 sqrt(流通市值)。"""
    y = g["excess_ret_fwd_10d"].to_numpy(dtype=np.float64, copy=False)
    f = g["factor_neutral_z"].to_numpy(dtype=np.float64, copy=False)
    dummies = industry_dummies(g["industry_level1"])
    X = np.column_stack([np.ones(len(g), dtype=np.float64), f, dummies])
    w = np.sqrt(g["float_market_cap"].to_numpy(dtype=np.float64, copy=False).clip(min=1.0))

    valid = np.isfinite(y) & np.isfinite(X).all(axis=1) & np.isfinite(w) & (w > 0)
    if valid.sum() < max(30, X.shape[1] + 5):
        return np.nan, np.nan

    Xv = X[valid]
    yv = y[valid]
    wv = w[valid]

    xtwx = Xv.T @ (wv[:, None] * Xv)
    xtwy = Xv.T @ (wv * yv)
    xtwx_inv = np.linalg.pinv(xtwx, rcond=1e-12)
    beta = xtwx_inv @ xtwy

    resid = yv - Xv @ beta
    rank = np.linalg.matrix_rank(xtwx)
    dof = max(len(yv) - rank, 1)
    sigma2 = float(np.sum(wv * resid * resid) / dof)
    se = np.sqrt(np.maximum(np.diag(sigma2 * xtwx_inv), 0.0))

    factor_ret = float(beta[1])
    t_value = float(beta[1] / se[1]) if se[1] > 1e-18 else np.nan
    return factor_ret, t_value


def calc_cross_section_metrics(g: pd.DataFrame, min_n: int) -> Optional[Dict[str, float]]:
    if len(g) < min_n:
        return None

    factor = g["factor_neutral_z"].to_numpy(dtype=np.float64, copy=False)
    ret = g["ret_fwd_10d"].to_numpy(dtype=np.float64, copy=False)
    valid = np.isfinite(factor) & np.isfinite(ret)
    if valid.sum() < min_n:
        return None

    ic = corr_np(factor[valid], ret[valid])
    rank_ic = corr_np(rank_np(factor[valid]), rank_np(ret[valid]))
    factor_ret, t_value = wls_factor_return(g)

    return {
        "date": g["date"].iloc[0],
        "IC": ic,
        "RankIC": rank_ic,
        "因子收益率": factor_ret,
        "t值": t_value,
    }


def calc_all_metrics(data: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    rows: List[Dict[str, float]] = []
    for _, g in data.groupby("date", sort=True, observed=True):
        row = calc_cross_section_metrics(g, cfg.min_cross_section_size)
        if row is not None:
            rows.append(row)
    if not rows:
        raise ValueError("没有足够截面可计算指标，请检查区间、股票池或最小截面样本数。")
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)


def safe_ir(s: pd.Series) -> float:
    s = pd.to_numeric(s, errors="coerce").dropna()
    std = s.std(ddof=1)
    if len(s) < 2 or not np.isfinite(std) or std <= 1e-18:
        return np.nan
    return float(s.mean() / std)


def make_summary(metrics: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    summary = pd.DataFrame([{
        "因子": cfg.factor_name,
        "起始日": metrics["date"].min().strftime("%Y-%m-%d"),
        "结束日": metrics["date"].max().strftime("%Y-%m-%d"),
        "截面数": int(metrics["date"].nunique()),
        "IC均值": metrics["IC"].mean(),
        "ICIR": safe_ir(metrics["IC"]),
        "RankIC均值": metrics["RankIC"].mean(),
        "RankICIR": safe_ir(metrics["RankIC"]),
        "因子收益率": metrics["因子收益率"].mean(),
        "t值": metrics["t值"].mean(),
    }])
    return summary


def format_summary(summary: pd.DataFrame) -> pd.DataFrame:
    out = summary.copy()
    decimal_cols = ["IC均值", "ICIR", "RankIC均值", "RankICIR", "因子收益率", "t值"]
    for col in decimal_cols:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{x:.6f}")
    return out


def plot_ic_rankic(metrics: pd.DataFrame, cfg: Config) -> None:
    font_prop = set_chinese_font(cfg.chinese_font_path)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(metrics["date"], metrics["IC"], label="IC", linewidth=1.6)
    ax.plot(metrics["date"], metrics["RankIC"], label="RankIC", linewidth=1.6)
    ax.axhline(0, linewidth=1.0, linestyle="--")

    title = f"{cfg.factor_name}：IC 与 RankIC 时序图"
    if font_prop is not None:
        ax.set_title(title, fontproperties=font_prop)
        ax.set_xlabel("日期", fontproperties=font_prop)
        ax.set_ylabel("相关系数", fontproperties=font_prop)
        ax.legend(prop=font_prop)
    else:
        ax.set_title(title)
        ax.set_xlabel("日期")
        ax.set_ylabel("相关系数")
        ax.legend()

    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def main() -> Tuple[pd.DataFrame, pd.DataFrame]:
    data = fetch_signal_panel(CFG)
    data = add_neutralized_factor(data)
    metrics = calc_all_metrics(data, CFG)
    summary = make_summary(metrics, CFG)

    display(format_summary(summary))
    plot_ic_rankic(metrics, CFG)
    return summary, metrics


summary, metrics = main()


从结果看，市值行业中性化后的 mfd_sellamt_d_neutral 仍然是一个有效性较明确的短周期资金流因子：在 2020-01-02 至 2026-06-15 的 157 个 10日截面中，IC 均值为 0.049053，RankIC 均值达到 0.080118，说明因子对未来 10 日收益具有比较稳定的正向预测能力，而且排序能力明显强于线性相关能力；ICIR 为 0.561582，RankICIR 为 0.958547，表明该因子的截面排序信号相对更稳健，适合用于分层选股或组合打分，而不一定适合直接用线性暴露解释收益；因子收益率均值为 0.002383，t 值为 2.681197，说明其在回归意义上也具备统计显著性，经过市值和行业中性化后仍然保留了有效信息，意味着它并不是单纯依赖小市值效应或行业暴露获得收益。不过从时序图看，IC 和 RankIC 波动并不小，部分阶段会出现明显负值，尤其在市场风格急剧切换或资金流行为失效的阶段可能会拖累组合，因此这个因子更适合作为多因子模型中的一类交易行为/资金流向增量信号，而不适合作为单独裸用的核心选股因子；整体评价是：该因子方向清晰、统计显著、RankIC 表现较好，具有实用价值，但需要与质量、动量、波动率、换手率等因子结合，并配合行业、市值、流动性和换手约束后使用。

## 市值分层回测

In [ ]:
# -*- coding: utf-8 -*-
"""
BigQuant 策略回测：mfd_sellamt_d_neutral 市值分层选股策略

因子口径：
1. mfd_sellamt_d ≈ cn_stock_moneyflow.outflow_amount_main。
2. 流出类因子为反向因子，原始因子取 -outflow_amount_main。
3. 每个信号截面：原始因子先做 MAD 去极值 + 标准化，再对 log(流通市值) 与行业做截面中性化，
   最后对中性化残差再次做 MAD 去极值 + 标准化，得到 mfd_sellamt_d_neutral。

策略逻辑：
1. 每 10 个交易日生成一次信号，信号使用 signal_date 当日已经可得的数据。
2. 全市场按流通市值从小到大划分 15 组，1=最小市值组，15=最大市值组。
3. 通过 SIZE_GROUPS_TO_TRADE 指定参与交易的市值组，例如 [1, 2, 3]。
4. 在每个指定市值组内，选取 mfd_sellamt_d_neutral 最高的前 10% 股票。
5. 所有入选股票等权配置。
6. 信号日后一交易日执行调仓；不在选股阶段读取执行日行情，避免未来函数。
7. 执行日使用当日开盘涨跌停状态做交易约束：开盘涨停不买入，开盘跌停不卖出；停牌不交易。
8. 股票池剔除 ST、*ST、停牌、北交所；考虑交易成本；使用 BigTrader 原生回测引擎。

注意：
- 执行日的开盘涨跌停限制只在 handle_data 当日下单时使用，不参与因子计算与选股。
- 如果某只股票因涨停无法买入或因跌停无法卖出，不做强制再平衡，避免使用执行日信息重新分配权重。
"""

import gc
import time
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import dai
except Exception as e:
    raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 dai。") from e

try:
    from bigquant import bigtrader
except Exception:
    try:
        import bigtrader
    except Exception as e:
        raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 BigTrader。") from e

warnings.filterwarnings("ignore")


# =========================
# 1. 参数设置
# =========================

START_DATE = "2020-01-01"
END_DATE = "2026-06-30"

FACTOR_SOURCE_COL = "outflow_amount_main"
RAW_FACTOR_NAME = "mfd_sellamt_d_raw"
NEUTRAL_FACTOR_NAME = "mfd_sellamt_d_neutral"

# 1=最小市值组，15=最大市值组
SIZE_GROUPS_TO_TRADE = [1, 2, 3, 4]
N_SIZE_GROUPS = 15
TOP_PCT = 0.10
REBALANCE_DAYS = 5
MIN_STOCKS_PER_SIZE_GROUP = 20
SELECT_COUNT_METHOD = "floor"  # floor 或 ceil；至少选 1 只

# 中性化与标准化
WINSOR_MAD_N = 3.0
MIN_CROSS_SECTION_SIZE = 100
INDUSTRY_COL = "sw2021_level1"  # 可改 sw2014_level1 / cs_level1

# 回测参数
CAPITAL_BASE = 1_000_000
BENCHMARK = "000300.SH"  # 如账号环境使用 CSI 代码，可改为 000300.SH / 000300.SH / 932000.CSI 等
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COMMISSION = 5

# 执行日涨跌停判断容忍误差
LIMIT_EPS = 1e-4

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")

if not SIZE_GROUPS_TO_TRADE:
    raise ValueError("SIZE_GROUPS_TO_TRADE 不能为空。")
SIZE_GROUPS_TO_TRADE = sorted(set(int(x) for x in SIZE_GROUPS_TO_TRADE))
bad_groups = [g for g in SIZE_GROUPS_TO_TRADE if g < 1 or g > N_SIZE_GROUPS]
if bad_groups:
    raise ValueError(f"SIZE_GROUPS_TO_TRADE 中存在非法市值组：{bad_groups}，有效范围为 1~{N_SIZE_GROUPS}。")
if SELECT_COUNT_METHOD not in {"floor", "ceil"}:
    raise ValueError("SELECT_COUNT_METHOD 只能是 'floor' 或 'ceil'。")


# =========================
# 2. 通用工具函数
# =========================

_T0 = time.time()


def _elapsed() -> str:
    sec = int(time.time() - _T0)
    return f"{sec // 60:02d}:{sec % 60:02d}"


def progress(msg: str) -> None:
    print(f"[{_elapsed()}] {msg}", flush=True)


def query_df(sql: str, filters: Optional[dict] = None) -> pd.DataFrame:
    if filters is None:
        return dai.query(sql).df()
    return dai.query(sql, filters=filters).df()


def date_in_sql(dates) -> str:
    return ", ".join(f"'{pd.to_datetime(x).strftime('%Y-%m-%d')}'" for x in dates)


def instrument_in_sql(instruments: List[str]) -> str:
    return ", ".join(f"'{str(x)}'" for x in instruments)


def downcast_float(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    return df


def robust_zscore_np(x: np.ndarray, mad_n: float = 3.0) -> np.ndarray:
    """截面 MAD 去极值后标准化。"""
    x = np.asarray(x, dtype=np.float64)
    out = np.full(x.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(x)
    if valid.sum() < 3:
        return out

    xv = x[valid]
    med = np.nanmedian(xv)
    mad = np.nanmedian(np.abs(xv - med))
    if np.isfinite(mad) and mad > 1e-12:
        scale = 1.4826 * mad
        lo, hi = med - mad_n * scale, med + mad_n * scale
    else:
        lo, hi = np.nanpercentile(xv, [1.0, 99.0])

    xv = np.clip(xv, lo, hi)
    std = xv.std(ddof=0)
    if np.isfinite(std) and std > 1e-12:
        out[valid] = (xv - xv.mean()) / std
    return out


def industry_dummies(industry: pd.Series) -> np.ndarray:
    """行业哑变量，drop_first=True。"""
    codes = pd.Categorical(industry.astype(str)).codes
    n = len(codes)
    k = int(codes.max()) + 1
    if k <= 1:
        return np.empty((n, 0), dtype=np.float64)
    mat = np.zeros((n, k - 1), dtype=np.float64)
    rows = np.arange(n)
    mask = codes > 0
    mat[rows[mask], codes[mask] - 1] = 1.0
    return mat


def neutralize_by_size_industry(g: pd.DataFrame) -> pd.DataFrame:
    """单个截面内完成：原始因子去极值标准化 -> 市值行业中性化 -> 残差去极值标准化。"""
    g = g.copy().sort_values("instrument", kind="mergesort").reset_index(drop=True)
    if len(g) < MIN_CROSS_SECTION_SIZE:
        return pd.DataFrame()

    raw = g[RAW_FACTOR_NAME].to_numpy(dtype=np.float64, copy=False)
    factor_z = robust_zscore_np(raw, WINSOR_MAD_N)
    log_cap = np.log(g["float_market_cap"].to_numpy(dtype=np.float64, copy=False).clip(min=1.0))
    dummies = industry_dummies(g["industry_level1"].fillna("未知"))
    X = np.column_stack([np.ones(len(g), dtype=np.float64), log_cap, dummies])

    valid = np.isfinite(factor_z) & np.isfinite(X).all(axis=1)
    if valid.sum() < max(MIN_CROSS_SECTION_SIZE, X.shape[1] + 5):
        return pd.DataFrame()

    resid = np.full(len(g), np.nan, dtype=np.float64)
    beta = np.linalg.lstsq(X[valid], factor_z[valid], rcond=None)[0]
    resid[valid] = factor_z[valid] - X[valid] @ beta
    neutral_z = robust_zscore_np(resid, WINSOR_MAD_N)

    out = g[["date", "instrument", "float_market_cap"]].copy()
    out[NEUTRAL_FACTOR_NAME] = neutral_z.astype(np.float32)
    out = out.dropna(subset=[NEUTRAL_FACTOR_NAME, "float_market_cap"])
    return out


def calc_select_count(n: int, pct: float) -> int:
    if SELECT_COUNT_METHOD == "ceil":
        return max(1, int(np.ceil(n * pct)))
    return max(1, int(np.floor(n * pct)))


def get_current_date_from_engine(context, data) -> Optional[str]:
    if data is not None and hasattr(data, "current_dt"):
        try:
            return pd.to_datetime(data.current_dt).strftime("%Y-%m-%d")
        except Exception:
            pass
    for attr in ["current_dt", "now", "current_date"]:
        if hasattr(context, attr):
            try:
                v = getattr(context, attr)
                if v is not None:
                    return pd.to_datetime(v).strftime("%Y-%m-%d")
            except Exception:
                pass
    return None


def get_positions_dict(context) -> Dict:
    for method in ["get_positions", "get_account_positions"]:
        if hasattr(context, method):
            try:
                pos = getattr(context, method)()
                if pos is not None:
                    return pos
            except Exception:
                pass
    try:
        return context.portfolio.positions
    except Exception:
        return {}


def position_amount(pos_obj) -> float:
    for attr in ["amount", "quantity", "volume", "position"]:
        try:
            return float(getattr(pos_obj, attr))
        except Exception:
            pass
    try:
        return float(pos_obj.get("amount", 0))
    except Exception:
        return 0.0


def position_market_value(pos_obj) -> float:
    for attr in ["market_value", "value"]:
        try:
            return float(getattr(pos_obj, attr))
        except Exception:
            pass
    try:
        return float(pos_obj.get("market_value", 0))
    except Exception:
        return 0.0


def portfolio_value(context) -> float:
    for attr in ["portfolio_value", "total_value", "market_value"]:
        try:
            v = float(getattr(context.portfolio, attr))
            if np.isfinite(v) and v > 0:
                return v
        except Exception:
            pass
    return np.nan


def order_to_target_percent(context, instrument: str, weight: float) -> bool:
    for method in ["order_target_percent", "order_percent"]:
        if hasattr(context, method):
            try:
                getattr(context, method)(instrument, float(weight))
                return True
            except Exception:
                continue
    print(f"下单失败：找不到可用的目标仓位下单函数，{instrument}, target={weight:.6f}", flush=True)
    return False


# =========================
# 3. 交易日、信号日、执行日
# =========================

progress("开始获取交易日与调仓日期")
trade_dates_sql = f"""
SELECT DISTINCT date
FROM cn_stock_factors_base
WHERE date >= DATE '{START_DATE}'
  AND date <= DATE '{END_DATE}'
ORDER BY date
"""
trade_dates_df = query_df(trade_dates_sql, filters={"date": [START_DATE, END_DATE]})
trade_dates = pd.to_datetime(trade_dates_df["date"]).drop_duplicates().sort_values().reset_index(drop=True)
if len(trade_dates) < REBALANCE_DAYS + 2:
    raise ValueError("指定时间段内交易日过少，无法完成回测。")

signal_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()
signal_to_execution = {}
for dt in signal_dates:
    idx_arr = trade_dates[trade_dates == dt].index
    if len(idx_arr) == 0:
        continue
    next_idx = int(idx_arr[0]) + 1
    if next_idx < len(trade_dates):
        signal_to_execution[pd.to_datetime(dt).strftime("%Y-%m-%d")] = pd.to_datetime(trade_dates.iloc[next_idx]).strftime("%Y-%m-%d")

if not signal_to_execution:
    raise ValueError("没有可用的信号日/执行日映射。")

signal_dates = [pd.to_datetime(x) for x in signal_to_execution.keys()]
execution_dates = [pd.to_datetime(x) for x in signal_to_execution.values()]
signal_date_sql = date_in_sql(signal_dates)
execution_date_sql = date_in_sql(execution_dates)

progress(f"交易日数量：{len(trade_dates):,}；信号截面数量：{len(signal_dates):,}；调仓周期：{REBALANCE_DAYS} 个交易日")
progress(f"参与交易市值组：{SIZE_GROUPS_TO_TRADE}；每组选择因子最高前 {TOP_PCT:.2%}")


# =========================
# 4. 读取信号截面数据
# =========================

progress("开始读取信号截面数据")

signal_sql = f"""
SELECT
    b.date,
    b.instrument,
    b.float_market_cap,
    b.{INDUSTRY_COL} AS industry_level1,
    -mf.{FACTOR_SOURCE_COL} AS {RAW_FACTOR_NAME}
FROM cn_stock_factors_base AS b
JOIN cn_stock_moneyflow AS mf
  ON b.date = mf.date AND b.instrument = mf.instrument
WHERE b.date IN ({signal_date_sql})
  AND b.list_sector != 4
  AND b.st_status = 0
  AND b.suspended = 0
  AND b.float_market_cap > 0
  AND mf.{FACTOR_SOURCE_COL} IS NOT NULL
ORDER BY b.date, b.instrument
"""

signal_panel = query_df(signal_sql, filters={"date": [START_DATE, END_DATE]})
if signal_panel.empty:
    raise ValueError("信号截面数据为空，请检查日期、字段名或数据权限。")

signal_panel["date"] = pd.to_datetime(signal_panel["date"]).dt.normalize()
signal_panel["instrument"] = signal_panel["instrument"].astype(str)
signal_panel["industry_level1"] = signal_panel["industry_level1"].fillna("未知").astype(str)
signal_panel = downcast_float(signal_panel, ["float_market_cap", RAW_FACTOR_NAME])
signal_panel = signal_panel.dropna(subset=["date", "instrument", "float_market_cap", RAW_FACTOR_NAME])
signal_panel = signal_panel[(signal_panel["float_market_cap"] > 0) & np.isfinite(signal_panel[RAW_FACTOR_NAME])]
signal_panel = signal_panel.drop_duplicates(subset=["date", "instrument"], keep="last")
progress(f"信号截面数据：{len(signal_panel):,} 行")


# =========================
# 5. 市值行业中性化、市值15组、组内Top 10%选股
# =========================

progress("开始逐截面中性化、市值分层与选股")
selected_parts = []
for i, (dt, g) in enumerate(signal_panel.groupby("date", sort=True), 1):
    if i == 1 or i % 10 == 0 or i == signal_panel["date"].nunique():
        progress(f"处理截面 {i}/{signal_panel['date'].nunique()}：{pd.to_datetime(dt).strftime('%Y-%m-%d')}，样本 {len(g):,}")

    neu = neutralize_by_size_industry(g)
    if neu.empty:
        continue

    neu = neu.sort_values(["float_market_cap", "instrument"], ascending=[True, True], kind="mergesort").reset_index(drop=True)
    rank = neu["float_market_cap"].rank(method="first", ascending=True)
    try:
        neu["size_group"] = pd.qcut(rank, q=N_SIZE_GROUPS, labels=list(range(1, N_SIZE_GROUPS + 1))).astype(int)
    except Exception:
        continue

    group_selected = []
    for sg_id in SIZE_GROUPS_TO_TRADE:
        sg = neu[neu["size_group"] == sg_id].copy()
        if len(sg) < MIN_STOCKS_PER_SIZE_GROUP:
            continue
        n_select = calc_select_count(len(sg), TOP_PCT)
        sg = sg.sort_values([NEUTRAL_FACTOR_NAME, "instrument"], ascending=[False, True], kind="mergesort")
        group_selected.append(sg.head(n_select))

    if group_selected:
        selected_parts.append(pd.concat(group_selected, ignore_index=True))

if not selected_parts:
    raise ValueError("没有形成任何有效选股结果，请检查市值组、样本数量或因子数据。")

selected_df = pd.concat(selected_parts, ignore_index=True)
del selected_parts, signal_panel
gc.collect()

selected_df["signal_date"] = selected_df["date"].dt.strftime("%Y-%m-%d")
selected_df["execution_date"] = selected_df["signal_date"].map(signal_to_execution)
selected_df = selected_df.dropna(subset=["execution_date"]).copy()
selected_df["stock_count"] = selected_df.groupby("signal_date")["instrument"].transform("count")
selected_df = selected_df[selected_df["stock_count"] > 0].copy()
selected_df["target_weight"] = 1.0 / selected_df["stock_count"]

signal_df = selected_df[["signal_date", "execution_date", "instrument", "size_group", NEUTRAL_FACTOR_NAME, "target_weight"]].copy()
signal_df = signal_df.sort_values(
    ["execution_date", "size_group", NEUTRAL_FACTOR_NAME, "instrument"],
    ascending=[True, True, False, True],
    kind="mergesort",
).reset_index(drop=True)

progress(f"最终信号：{len(signal_df):,} 行；涉及股票 {signal_df['instrument'].nunique():,} 只")

signal_summary = (
    signal_df.groupby(["signal_date", "execution_date"], sort=True)
    .agg(stock_count=("instrument", "count"), avg_weight=("target_weight", "mean"))
    .reset_index()
)
progress("交易信号摘要前20行：")
display(signal_summary.head(20))


# =========================
# 6. 读取执行日交易约束：开盘涨跌停、停牌
# =========================

progress("开始读取执行日交易约束")
trade_status_sql = f"""
SELECT
    date,
    instrument,
    open,
    upper_limit,
    lower_limit,
    suspended,
    st_status
FROM cn_stock_factors_base
WHERE date IN ({execution_date_sql})
ORDER BY date, instrument
"""
trade_status = query_df(trade_status_sql, filters={"date": [START_DATE, END_DATE]})
if trade_status.empty:
    raise ValueError("执行日交易约束数据为空，请检查 cn_stock_factors_base 字段或日期。")

trade_status["date"] = pd.to_datetime(trade_status["date"]).dt.strftime("%Y-%m-%d")
trade_status["instrument"] = trade_status["instrument"].astype(str)
trade_status = downcast_float(trade_status, ["open", "upper_limit", "lower_limit"])
trade_status["suspended"] = pd.to_numeric(trade_status["suspended"], errors="coerce").fillna(1).astype(int)
trade_status["st_status"] = pd.to_numeric(trade_status["st_status"], errors="coerce").fillna(0).astype(int)
trade_status = trade_status.drop_duplicates(subset=["date", "instrument"], keep="last")

valid_price = (
    np.isfinite(trade_status["open"]) &
    np.isfinite(trade_status["upper_limit"]) &
    np.isfinite(trade_status["lower_limit"]) &
    (trade_status["open"] > 0) &
    (trade_status["upper_limit"] > 0) &
    (trade_status["lower_limit"] > 0)
)
trade_status["open_limit_up"] = valid_price & (trade_status["open"] >= trade_status["upper_limit"] * (1.0 - LIMIT_EPS))
trade_status["open_limit_down"] = valid_price & (trade_status["open"] <= trade_status["lower_limit"] * (1.0 + LIMIT_EPS))
trade_status["can_buy_open"] = (trade_status["suspended"] == 0) & (~trade_status["open_limit_up"])
trade_status["can_sell_open"] = (trade_status["suspended"] == 0) & (~trade_status["open_limit_down"])

trade_status_by_date = {
    d: g.set_index("instrument")[["can_buy_open", "can_sell_open", "suspended", "open_limit_up", "open_limit_down"]].to_dict("index")
    for d, g in trade_status.groupby("date", sort=False)
}

del trade_status
gc.collect()


# =========================
# 7. BigTrader 原生回测
# =========================

progress("开始准备 BigTrader 回测输入")
backtest_data = signal_df[["execution_date", "instrument", "target_weight"]].copy()
backtest_data = backtest_data.rename(columns={"execution_date": "date"})
backtest_data["date"] = pd.to_datetime(backtest_data["date"]).dt.strftime("%Y-%m-%d")
backtest_data["instrument"] = backtest_data["instrument"].astype(str)

signal_by_execution_date = {
    d: g[["instrument", "target_weight"]].copy()
    for d, g in backtest_data.groupby("date", sort=True)
}

target_by_execution_date = {
    d: set(g["instrument"].astype(str))
    for d, g in backtest_data.groupby("date", sort=True)
}

progress("开始运行 BigTrader 原生回测")


def initialize(context):
    try:
        context.set_commission(
            bigtrader.PerOrder(
                buy_cost=BUY_COST,
                sell_cost=SELL_COST,
                min_cost=MIN_COMMISSION,
            )
        )
    except Exception as e:
        print(f"设置手续费失败，将使用引擎默认费率。原因：{e}", flush=True)

    context.signal_by_execution_date = signal_by_execution_date
    context.target_by_execution_date = target_by_execution_date
    context.trade_status_by_date = trade_status_by_date
    context.rebalance_dates = set(signal_by_execution_date.keys())

    try:
        context.subscribe_bar(list(backtest_data["instrument"].drop_duplicates()), "1d", None)
    except Exception:
        pass


def _get_trade_flags(context, current_date: str, instrument: str) -> Tuple[bool, bool]:
    row = context.trade_status_by_date.get(current_date, {}).get(str(instrument))
    if row is None:
        return False, False
    return bool(row.get("can_buy_open", False)), bool(row.get("can_sell_open", False))


def handle_data(context, data):
    current_date = get_current_date_from_engine(context, data)
    if current_date is None or current_date not in context.rebalance_dates:
        return

    today_signal = context.signal_by_execution_date.get(current_date)
    if today_signal is None or len(today_signal) == 0:
        return

    target_weights = dict(zip(today_signal["instrument"].astype(str), today_signal["target_weight"].astype(float)))
    target_instruments = set(target_weights.keys())
    positions = get_positions_dict(context)

    holding_instruments = set()
    current_weights = {}
    pv = portfolio_value(context)
    for ins, pos in positions.items():
        ins = str(ins)
        amt = position_amount(pos)
        if amt <= 0:
            continue
        holding_instruments.add(ins)
        mv = position_market_value(pos)
        if np.isfinite(pv) and pv > 0 and np.isfinite(mv):
            current_weights[ins] = mv / pv

    # 先卖出不在目标池中的股票；开盘跌停或停牌则不卖。
    for ins in sorted(holding_instruments - target_instruments):
        _, can_sell = _get_trade_flags(context, current_date, ins)
        if can_sell:
            order_to_target_percent(context, ins, 0.0)

    # 再调整目标股票。买入方向要求非开盘涨停；卖出方向要求非开盘跌停。
    for ins in sorted(target_weights.keys()):
        target_w = float(target_weights[ins])
        can_buy, can_sell = _get_trade_flags(context, current_date, ins)
        cur_w = current_weights.get(ins, 0.0)

        if cur_w <= 1e-8:
            if can_buy:
                order_to_target_percent(context, ins, target_w)
        elif target_w > cur_w + 1e-5:
            if can_buy:
                order_to_target_percent(context, ins, target_w)
        elif target_w < cur_w - 1e-5:
            if can_sell:
                order_to_target_percent(context, ins, target_w)
        else:
            # 已接近目标权重，不下单。
            pass


run_kwargs = dict(
    data=backtest_data,
    start_date=min(signal_by_execution_date.keys()),
    end_date=END_DATE,
    initialize=initialize,
    handle_data=handle_data,
    capital_base=CAPITAL_BASE,
    benchmark=BENCHMARK,
)

try:
    run_kwargs["market"] = bigtrader.Market.CN_STOCK
except Exception:
    pass

try:
    run_kwargs["frequency"] = bigtrader.Frequency.DAILY
except Exception:
    run_kwargs["frequency"] = "1d"

performance = bigtrader.run(**run_kwargs)

progress("BigTrader 回测完成")
try:
    display(performance.summary)
except Exception:
    display(performance)


结合多次测试的输出结果，发现选取市值组1，2，3，4，在每组中取因子值最高的10%的股票，5日调仓等权购买所得到的结果是最好的

## N日累计主力流出因子市值分层回测

注意到原因子只对当日的主力流出进行计算，但现实市场中的环境是一支股票可能在最近存在累计主力流出，所以考虑将该因子扩展为N日累计主力流出因子再进行测试，与原因子进行比较

In [ ]:
# -*- coding: utf-8 -*-
"""
BigQuant 策略回测：N日累计主力流出额中性化因子市值分层选股策略

因子口径：
1. mfd_sellamt_d ≈ cn_stock_moneyflow.outflow_amount_main。
2. 流出类因子为反向因子，N日累计原始因子取 -sum(outflow_amount_main, N日)。
3. N日窗口包含信号日当日及过去 N-1 个交易日，即 [T-(N-1), T]。
4. 每个信号截面：N日累计原始因子先做 MAD 去极值 + 标准化，再对 log(流通市值) 与行业做截面中性化，
   最后对中性化残差再次做 MAD 去极值 + 标准化，得到 mfd_sellamt_d_sum_N_neutral。

策略逻辑：
1. 每 10 个交易日生成一次信号，信号使用 signal_date 当日已经可得的数据。
2. 全市场按流通市值从小到大划分 15 组，1=最小市值组，15=最大市值组。
3. 通过 SIZE_GROUPS_TO_TRADE 指定参与交易的市值组，例如 [1, 2, 3]。
4. 在每个指定市值组内，选取 N日累计主力流出中性化因子最高的前 10% 股票。
5. 所有入选股票等权配置。
6. 信号日后一交易日执行调仓；不在选股阶段读取执行日行情，避免未来函数。
7. 执行日使用当日开盘涨跌停状态做交易约束：开盘涨停不买入，开盘跌停不卖出；停牌不交易。
8. 股票池剔除 ST、*ST、停牌、北交所；考虑交易成本；使用 BigTrader 原生回测引擎。

注意：
- 执行日的开盘涨跌停限制只在 handle_data 当日下单时使用，不参与因子计算与选股。
- 如果某只股票因涨停无法买入或因跌停无法卖出，不做强制再平衡，避免使用执行日信息重新分配权重。
"""

import gc
import time
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import dai
except Exception as e:
    raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 dai。") from e

try:
    from bigquant import bigtrader
except Exception:
    try:
        import bigtrader
    except Exception as e:
        raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 BigTrader。") from e

warnings.filterwarnings("ignore")


# =========================
# 1. 参数设置
# =========================

START_DATE = "2020-01-01"
END_DATE = "2026-06-30"

FACTOR_SOURCE_COL = "outflow_amount_main"
FACTOR_WINDOW_DAYS = 15  # N日累计窗口，包含信号日当日及过去 N-1 个交易日，即 [T-(N-1), T]
RAW_FACTOR_NAME = f"mfd_sellamt_d_sum_{FACTOR_WINDOW_DAYS}_raw"
NEUTRAL_FACTOR_NAME = f"mfd_sellamt_d_sum_{FACTOR_WINDOW_DAYS}_neutral"

# 1=最小市值组，15=最大市值组
SIZE_GROUPS_TO_TRADE = [1, 2, 3, 4]
N_SIZE_GROUPS = 15
TOP_PCT = 0.10
REBALANCE_DAYS = 5
MIN_STOCKS_PER_SIZE_GROUP = 20
SELECT_COUNT_METHOD = "floor"  # floor 或 ceil；至少选 1 只

# 中性化与标准化
WINSOR_MAD_N = 3.0
MIN_CROSS_SECTION_SIZE = 100
INDUSTRY_COL = "sw2021_level1"  # 可改 sw2014_level1 / cs_level1

# 回测参数
CAPITAL_BASE = 1_000_000
BENCHMARK = "000300.SH"  # 如账号环境使用 CSI 代码，可改为 000300.SH / 000300.SH / 932000.CSI 等
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COMMISSION = 5

# 执行日涨跌停判断容忍误差
LIMIT_EPS = 1e-4

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")

if int(FACTOR_WINDOW_DAYS) < 1:
    raise ValueError("FACTOR_WINDOW_DAYS 必须为正整数。")
FACTOR_WINDOW_DAYS = int(FACTOR_WINDOW_DAYS)
# 为滚动N个交易日窗口预留足够自然日；真正的N日窗口在SQL中用 ROWS BETWEEN N-1 PRECEDING AND CURRENT ROW 实现。
QUERY_START_DATE = (pd.to_datetime(START_DATE) - pd.Timedelta(days=max(60, FACTOR_WINDOW_DAYS * 4))).strftime("%Y-%m-%d")

if not SIZE_GROUPS_TO_TRADE:
    raise ValueError("SIZE_GROUPS_TO_TRADE 不能为空。")
SIZE_GROUPS_TO_TRADE = sorted(set(int(x) for x in SIZE_GROUPS_TO_TRADE))
bad_groups = [g for g in SIZE_GROUPS_TO_TRADE if g < 1 or g > N_SIZE_GROUPS]
if bad_groups:
    raise ValueError(f"SIZE_GROUPS_TO_TRADE 中存在非法市值组：{bad_groups}，有效范围为 1~{N_SIZE_GROUPS}。")
if SELECT_COUNT_METHOD not in {"floor", "ceil"}:
    raise ValueError("SELECT_COUNT_METHOD 只能是 'floor' 或 'ceil'。")


# =========================
# 2. 通用工具函数
# =========================

_T0 = time.time()


def _elapsed() -> str:
    sec = int(time.time() - _T0)
    return f"{sec // 60:02d}:{sec % 60:02d}"


def progress(msg: str) -> None:
    print(f"[{_elapsed()}] {msg}", flush=True)


def query_df(sql: str, filters: Optional[dict] = None) -> pd.DataFrame:
    if filters is None:
        return dai.query(sql).df()
    return dai.query(sql, filters=filters).df()


def date_in_sql(dates) -> str:
    return ", ".join(f"'{pd.to_datetime(x).strftime('%Y-%m-%d')}'" for x in dates)


def instrument_in_sql(instruments: List[str]) -> str:
    return ", ".join(f"'{str(x)}'" for x in instruments)


def downcast_float(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    return df


def robust_zscore_np(x: np.ndarray, mad_n: float = 3.0) -> np.ndarray:
    """截面 MAD 去极值后标准化。"""
    x = np.asarray(x, dtype=np.float64)
    out = np.full(x.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(x)
    if valid.sum() < 3:
        return out

    xv = x[valid]
    med = np.nanmedian(xv)
    mad = np.nanmedian(np.abs(xv - med))
    if np.isfinite(mad) and mad > 1e-12:
        scale = 1.4826 * mad
        lo, hi = med - mad_n * scale, med + mad_n * scale
    else:
        lo, hi = np.nanpercentile(xv, [1.0, 99.0])

    xv = np.clip(xv, lo, hi)
    std = xv.std(ddof=0)
    if np.isfinite(std) and std > 1e-12:
        out[valid] = (xv - xv.mean()) / std
    return out


def industry_dummies(industry: pd.Series) -> np.ndarray:
    """行业哑变量，drop_first=True。"""
    codes = pd.Categorical(industry.astype(str)).codes
    n = len(codes)
    k = int(codes.max()) + 1
    if k <= 1:
        return np.empty((n, 0), dtype=np.float64)
    mat = np.zeros((n, k - 1), dtype=np.float64)
    rows = np.arange(n)
    mask = codes > 0
    mat[rows[mask], codes[mask] - 1] = 1.0
    return mat


def neutralize_by_size_industry(g: pd.DataFrame) -> pd.DataFrame:
    """单个截面内完成：原始因子去极值标准化 -> 市值行业中性化 -> 残差去极值标准化。"""
    g = g.copy().sort_values("instrument", kind="mergesort").reset_index(drop=True)
    if len(g) < MIN_CROSS_SECTION_SIZE:
        return pd.DataFrame()

    raw = g[RAW_FACTOR_NAME].to_numpy(dtype=np.float64, copy=False)
    factor_z = robust_zscore_np(raw, WINSOR_MAD_N)
    log_cap = np.log(g["float_market_cap"].to_numpy(dtype=np.float64, copy=False).clip(min=1.0))
    dummies = industry_dummies(g["industry_level1"].fillna("未知"))
    X = np.column_stack([np.ones(len(g), dtype=np.float64), log_cap, dummies])

    valid = np.isfinite(factor_z) & np.isfinite(X).all(axis=1)
    if valid.sum() < max(MIN_CROSS_SECTION_SIZE, X.shape[1] + 5):
        return pd.DataFrame()

    resid = np.full(len(g), np.nan, dtype=np.float64)
    beta = np.linalg.lstsq(X[valid], factor_z[valid], rcond=None)[0]
    resid[valid] = factor_z[valid] - X[valid] @ beta
    neutral_z = robust_zscore_np(resid, WINSOR_MAD_N)

    out = g[["date", "instrument", "float_market_cap"]].copy()
    out[NEUTRAL_FACTOR_NAME] = neutral_z.astype(np.float32)
    out = out.dropna(subset=[NEUTRAL_FACTOR_NAME, "float_market_cap"])
    return out


def calc_select_count(n: int, pct: float) -> int:
    if SELECT_COUNT_METHOD == "ceil":
        return max(1, int(np.ceil(n * pct)))
    return max(1, int(np.floor(n * pct)))


def get_current_date_from_engine(context, data) -> Optional[str]:
    if data is not None and hasattr(data, "current_dt"):
        try:
            return pd.to_datetime(data.current_dt).strftime("%Y-%m-%d")
        except Exception:
            pass
    for attr in ["current_dt", "now", "current_date"]:
        if hasattr(context, attr):
            try:
                v = getattr(context, attr)
                if v is not None:
                    return pd.to_datetime(v).strftime("%Y-%m-%d")
            except Exception:
                pass
    return None


def get_positions_dict(context) -> Dict:
    for method in ["get_positions", "get_account_positions"]:
        if hasattr(context, method):
            try:
                pos = getattr(context, method)()
                if pos is not None:
                    return pos
            except Exception:
                pass
    try:
        return context.portfolio.positions
    except Exception:
        return {}


def position_amount(pos_obj) -> float:
    for attr in ["amount", "quantity", "volume", "position"]:
        try:
            return float(getattr(pos_obj, attr))
        except Exception:
            pass
    try:
        return float(pos_obj.get("amount", 0))
    except Exception:
        return 0.0


def position_market_value(pos_obj) -> float:
    for attr in ["market_value", "value"]:
        try:
            return float(getattr(pos_obj, attr))
        except Exception:
            pass
    try:
        return float(pos_obj.get("market_value", 0))
    except Exception:
        return 0.0


def portfolio_value(context) -> float:
    for attr in ["portfolio_value", "total_value", "market_value"]:
        try:
            v = float(getattr(context.portfolio, attr))
            if np.isfinite(v) and v > 0:
                return v
        except Exception:
            pass
    return np.nan


def order_to_target_percent(context, instrument: str, weight: float) -> bool:
    for method in ["order_target_percent", "order_percent"]:
        if hasattr(context, method):
            try:
                getattr(context, method)(instrument, float(weight))
                return True
            except Exception:
                continue
    print(f"下单失败：找不到可用的目标仓位下单函数，{instrument}, target={weight:.6f}", flush=True)
    return False


# =========================
# 3. 交易日、信号日、执行日
# =========================

progress("开始获取交易日与调仓日期")
trade_dates_sql = f"""
SELECT DISTINCT date
FROM cn_stock_factors_base
WHERE date >= DATE '{START_DATE}'
  AND date <= DATE '{END_DATE}'
ORDER BY date
"""
trade_dates_df = query_df(trade_dates_sql, filters={"date": [START_DATE, END_DATE]})
trade_dates = pd.to_datetime(trade_dates_df["date"]).drop_duplicates().sort_values().reset_index(drop=True)
if len(trade_dates) < REBALANCE_DAYS + 2:
    raise ValueError("指定时间段内交易日过少，无法完成回测。")

signal_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()
signal_to_execution = {}
for dt in signal_dates:
    idx_arr = trade_dates[trade_dates == dt].index
    if len(idx_arr) == 0:
        continue
    next_idx = int(idx_arr[0]) + 1
    if next_idx < len(trade_dates):
        signal_to_execution[pd.to_datetime(dt).strftime("%Y-%m-%d")] = pd.to_datetime(trade_dates.iloc[next_idx]).strftime("%Y-%m-%d")

if not signal_to_execution:
    raise ValueError("没有可用的信号日/执行日映射。")

signal_dates = [pd.to_datetime(x) for x in signal_to_execution.keys()]
execution_dates = [pd.to_datetime(x) for x in signal_to_execution.values()]
signal_date_sql = date_in_sql(signal_dates)
execution_date_sql = date_in_sql(execution_dates)

progress(f"交易日数量：{len(trade_dates):,}；信号截面数量：{len(signal_dates):,}；调仓周期：{REBALANCE_DAYS} 个交易日")
progress(f"累计窗口：{FACTOR_WINDOW_DAYS} 个交易日；参与交易市值组：{SIZE_GROUPS_TO_TRADE}；每组选择因子最高前 {TOP_PCT:.2%}")


# =========================
# 4. 读取信号截面数据
# =========================

progress("开始读取信号截面数据")

signal_sql = f"""
PRAGMA enable_pushdown_window;

WITH moneyflow_base AS (
    SELECT
        b.date,
        b.instrument,
        b.float_market_cap,
        b.{INDUSTRY_COL} AS industry_level1,
        b.list_sector,
        b.st_status,
        b.suspended,
        mf.{FACTOR_SOURCE_COL} AS source_outflow_amount
    FROM cn_stock_factors_base AS b
    JOIN cn_stock_moneyflow AS mf
      ON b.date = mf.date AND b.instrument = mf.instrument
    WHERE b.date >= DATE '{QUERY_START_DATE}'
      AND b.date <= DATE '{END_DATE}'
      AND b.list_sector != 4
      AND b.float_market_cap > 0
      AND mf.{FACTOR_SOURCE_COL} IS NOT NULL
),

moneyflow_roll AS (
    SELECT
        date,
        instrument,
        float_market_cap,
        industry_level1,
        list_sector,
        st_status,
        suspended,
        SUM(source_outflow_amount) OVER (
            PARTITION BY instrument
            ORDER BY date
            ROWS BETWEEN {FACTOR_WINDOW_DAYS - 1} PRECEDING AND CURRENT ROW
        ) AS source_outflow_amount_sum_n,
        COUNT(source_outflow_amount) OVER (
            PARTITION BY instrument
            ORDER BY date
            ROWS BETWEEN {FACTOR_WINDOW_DAYS - 1} PRECEDING AND CURRENT ROW
        ) AS source_obs_n
    FROM moneyflow_base
)

SELECT
    date,
    instrument,
    float_market_cap,
    industry_level1,
    -source_outflow_amount_sum_n AS {RAW_FACTOR_NAME}
FROM moneyflow_roll
WHERE date IN ({signal_date_sql})
  AND source_obs_n = {FACTOR_WINDOW_DAYS}
  AND st_status = 0
  AND suspended = 0
  AND float_market_cap > 0
  AND source_outflow_amount_sum_n IS NOT NULL
ORDER BY date, instrument
"""

signal_panel = query_df(signal_sql, filters={"date": [QUERY_START_DATE, END_DATE]})
if signal_panel.empty:
    raise ValueError("信号截面数据为空，请检查日期、字段名或数据权限。")

signal_panel["date"] = pd.to_datetime(signal_panel["date"]).dt.normalize()
signal_panel["instrument"] = signal_panel["instrument"].astype(str)
signal_panel["industry_level1"] = signal_panel["industry_level1"].fillna("未知").astype(str)
signal_panel = downcast_float(signal_panel, ["float_market_cap", RAW_FACTOR_NAME])
signal_panel = signal_panel.dropna(subset=["date", "instrument", "float_market_cap", RAW_FACTOR_NAME])
signal_panel = signal_panel[(signal_panel["float_market_cap"] > 0) & np.isfinite(signal_panel[RAW_FACTOR_NAME])]
signal_panel = signal_panel.drop_duplicates(subset=["date", "instrument"], keep="last")
progress(f"N日累计主力流出信号截面数据：{len(signal_panel):,} 行；原始因子列：{RAW_FACTOR_NAME}")


# =========================
# 5. N日累计因子市值行业中性化、市值15组、组内Top 10%选股
# =========================

progress("开始逐截面N日累计因子中性化、市值分层与选股")
selected_parts = []
for i, (dt, g) in enumerate(signal_panel.groupby("date", sort=True), 1):
    if i == 1 or i % 10 == 0 or i == signal_panel["date"].nunique():
        progress(f"处理截面 {i}/{signal_panel['date'].nunique()}：{pd.to_datetime(dt).strftime('%Y-%m-%d')}，样本 {len(g):,}")

    neu = neutralize_by_size_industry(g)
    if neu.empty:
        continue

    neu = neu.sort_values(["float_market_cap", "instrument"], ascending=[True, True], kind="mergesort").reset_index(drop=True)
    rank = neu["float_market_cap"].rank(method="first", ascending=True)
    try:
        neu["size_group"] = pd.qcut(rank, q=N_SIZE_GROUPS, labels=list(range(1, N_SIZE_GROUPS + 1))).astype(int)
    except Exception:
        continue

    group_selected = []
    for sg_id in SIZE_GROUPS_TO_TRADE:
        sg = neu[neu["size_group"] == sg_id].copy()
        if len(sg) < MIN_STOCKS_PER_SIZE_GROUP:
            continue
        n_select = calc_select_count(len(sg), TOP_PCT)
        sg = sg.sort_values([NEUTRAL_FACTOR_NAME, "instrument"], ascending=[False, True], kind="mergesort")
        group_selected.append(sg.head(n_select))

    if group_selected:
        selected_parts.append(pd.concat(group_selected, ignore_index=True))

if not selected_parts:
    raise ValueError("没有形成任何有效选股结果，请检查市值组、样本数量或因子数据。")

selected_df = pd.concat(selected_parts, ignore_index=True)
del selected_parts, signal_panel
gc.collect()

selected_df["signal_date"] = selected_df["date"].dt.strftime("%Y-%m-%d")
selected_df["execution_date"] = selected_df["signal_date"].map(signal_to_execution)
selected_df = selected_df.dropna(subset=["execution_date"]).copy()
selected_df["stock_count"] = selected_df.groupby("signal_date")["instrument"].transform("count")
selected_df = selected_df[selected_df["stock_count"] > 0].copy()
selected_df["target_weight"] = 1.0 / selected_df["stock_count"]

signal_df = selected_df[["signal_date", "execution_date", "instrument", "size_group", NEUTRAL_FACTOR_NAME, "target_weight"]].copy()
signal_df = signal_df.sort_values(
    ["execution_date", "size_group", NEUTRAL_FACTOR_NAME, "instrument"],
    ascending=[True, True, False, True],
    kind="mergesort",
).reset_index(drop=True)

progress(f"最终信号：{len(signal_df):,} 行；涉及股票 {signal_df['instrument'].nunique():,} 只")

signal_summary = (
    signal_df.groupby(["signal_date", "execution_date"], sort=True)
    .agg(stock_count=("instrument", "count"), avg_weight=("target_weight", "mean"))
    .reset_index()
)
progress("交易信号摘要前20行：")
display(signal_summary.head(20))


# =========================
# 6. 读取执行日交易约束：开盘涨跌停、停牌
# =========================

progress("开始读取执行日交易约束")
trade_status_sql = f"""
SELECT
    date,
    instrument,
    open,
    upper_limit,
    lower_limit,
    suspended,
    st_status
FROM cn_stock_factors_base
WHERE date IN ({execution_date_sql})
ORDER BY date, instrument
"""
trade_status = query_df(trade_status_sql, filters={"date": [START_DATE, END_DATE]})
if trade_status.empty:
    raise ValueError("执行日交易约束数据为空，请检查 cn_stock_factors_base 字段或日期。")

trade_status["date"] = pd.to_datetime(trade_status["date"]).dt.strftime("%Y-%m-%d")
trade_status["instrument"] = trade_status["instrument"].astype(str)
trade_status = downcast_float(trade_status, ["open", "upper_limit", "lower_limit"])
trade_status["suspended"] = pd.to_numeric(trade_status["suspended"], errors="coerce").fillna(1).astype(int)
trade_status["st_status"] = pd.to_numeric(trade_status["st_status"], errors="coerce").fillna(0).astype(int)
trade_status = trade_status.drop_duplicates(subset=["date", "instrument"], keep="last")

valid_price = (
    np.isfinite(trade_status["open"]) &
    np.isfinite(trade_status["upper_limit"]) &
    np.isfinite(trade_status["lower_limit"]) &
    (trade_status["open"] > 0) &
    (trade_status["upper_limit"] > 0) &
    (trade_status["lower_limit"] > 0)
)
trade_status["open_limit_up"] = valid_price & (trade_status["open"] >= trade_status["upper_limit"] * (1.0 - LIMIT_EPS))
trade_status["open_limit_down"] = valid_price & (trade_status["open"] <= trade_status["lower_limit"] * (1.0 + LIMIT_EPS))
trade_status["can_buy_open"] = (trade_status["suspended"] == 0) & (~trade_status["open_limit_up"])
trade_status["can_sell_open"] = (trade_status["suspended"] == 0) & (~trade_status["open_limit_down"])

trade_status_by_date = {
    d: g.set_index("instrument")[["can_buy_open", "can_sell_open", "suspended", "open_limit_up", "open_limit_down"]].to_dict("index")
    for d, g in trade_status.groupby("date", sort=False)
}

del trade_status
gc.collect()


# =========================
# 7. BigTrader 原生回测
# =========================

progress("开始准备 BigTrader 回测输入")
backtest_data = signal_df[["execution_date", "instrument", "target_weight"]].copy()
backtest_data = backtest_data.rename(columns={"execution_date": "date"})
backtest_data["date"] = pd.to_datetime(backtest_data["date"]).dt.strftime("%Y-%m-%d")
backtest_data["instrument"] = backtest_data["instrument"].astype(str)

signal_by_execution_date = {
    d: g[["instrument", "target_weight"]].copy()
    for d, g in backtest_data.groupby("date", sort=True)
}

target_by_execution_date = {
    d: set(g["instrument"].astype(str))
    for d, g in backtest_data.groupby("date", sort=True)
}

progress("开始运行 BigTrader 原生回测")


def initialize(context):
    try:
        context.set_commission(
            bigtrader.PerOrder(
                buy_cost=BUY_COST,
                sell_cost=SELL_COST,
                min_cost=MIN_COMMISSION,
            )
        )
    except Exception as e:
        print(f"设置手续费失败，将使用引擎默认费率。原因：{e}", flush=True)

    context.signal_by_execution_date = signal_by_execution_date
    context.target_by_execution_date = target_by_execution_date
    context.trade_status_by_date = trade_status_by_date
    context.rebalance_dates = set(signal_by_execution_date.keys())

    try:
        context.subscribe_bar(list(backtest_data["instrument"].drop_duplicates()), "1d", None)
    except Exception:
        pass


def _get_trade_flags(context, current_date: str, instrument: str) -> Tuple[bool, bool]:
    row = context.trade_status_by_date.get(current_date, {}).get(str(instrument))
    if row is None:
        return False, False
    return bool(row.get("can_buy_open", False)), bool(row.get("can_sell_open", False))


def handle_data(context, data):
    current_date = get_current_date_from_engine(context, data)
    if current_date is None or current_date not in context.rebalance_dates:
        return

    today_signal = context.signal_by_execution_date.get(current_date)
    if today_signal is None or len(today_signal) == 0:
        return

    target_weights = dict(zip(today_signal["instrument"].astype(str), today_signal["target_weight"].astype(float)))
    target_instruments = set(target_weights.keys())
    positions = get_positions_dict(context)

    holding_instruments = set()
    current_weights = {}
    pv = portfolio_value(context)
    for ins, pos in positions.items():
        ins = str(ins)
        amt = position_amount(pos)
        if amt <= 0:
            continue
        holding_instruments.add(ins)
        mv = position_market_value(pos)
        if np.isfinite(pv) and pv > 0 and np.isfinite(mv):
            current_weights[ins] = mv / pv

    # 先卖出不在目标池中的股票；开盘跌停或停牌则不卖。
    for ins in sorted(holding_instruments - target_instruments):
        _, can_sell = _get_trade_flags(context, current_date, ins)
        if can_sell:
            order_to_target_percent(context, ins, 0.0)

    # 再调整目标股票。买入方向要求非开盘涨停；卖出方向要求非开盘跌停。
    for ins in sorted(target_weights.keys()):
        target_w = float(target_weights[ins])
        can_buy, can_sell = _get_trade_flags(context, current_date, ins)
        cur_w = current_weights.get(ins, 0.0)

        if cur_w <= 1e-8:
            if can_buy:
                order_to_target_percent(context, ins, target_w)
        elif target_w > cur_w + 1e-5:
            if can_buy:
                order_to_target_percent(context, ins, target_w)
        elif target_w < cur_w - 1e-5:
            if can_sell:
                order_to_target_percent(context, ins, target_w)
        else:
            # 已接近目标权重，不下单。
            pass


run_kwargs = dict(
    data=backtest_data,
    start_date=min(signal_by_execution_date.keys()),
    end_date=END_DATE,
    initialize=initialize,
    handle_data=handle_data,
    capital_base=CAPITAL_BASE,
    benchmark=BENCHMARK,
)

try:
    run_kwargs["market"] = bigtrader.Market.CN_STOCK
except Exception:
    pass

try:
    run_kwargs["frequency"] = bigtrader.Frequency.DAILY
except Exception:
    run_kwargs["frequency"] = "1d"

performance = bigtrader.run(**run_kwargs)

progress("BigTrader 回测完成")
try:
    display(performance.summary)
except Exception:
    display(performance)


通过将因子扩展成N日累计主力流出因子，我们将组合策略的年化收益提高到了27.33%，夏普比率提高到了0.87，胜率提高到了63.46%，可以说是有了较为显著的提升，这一步对因子的处理是有效的